# 9.4 Other features of data statements

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Additional features of the AMPL `data` format are provided to handle special situations.
We describe here the `data` statements that specify default values for parameters, that
define the membership of individual sets within an indexed collection of sets, and that
assign initial values to variables.

Default values
Data statements must provide values for exactly the parameters in your `model`. You
will receive an error message if you give a value for a nonexistent parameter:

```ampl
error processing param cost:
        invalid subscript cost['PITT','DET','coils'] discarded.
```

or if you fail to give a value for a parameter that does exist:

```ampl
error processing objective Total_Cost:
        no value for cost['CLEV','LAN','coils']
```

The error message appears the first time that AMPL tries to use the offending parameter,
usually after you type `solve`.

If the same value would appear many times in a `data` statement, you can avoid specifying
it repeatedly by including a default phrase that provides the value to be used
when no explicit value is given. For example, suppose that the parameter cost above is
indexed over all possible triples:

```ampl
set ORIG;
set DEST;
set PROD;
param cost {ORIG,DEST,PROD} >= 0;
```

but that a very high cost is assigned to routes that should not be used. This can be
expressed as

```ampl
param cost default 9999 :=
 [*,*,bands]: FRA DET LAN WIN   STL   FRE   LAF :=
        CLEV   27   9   12  .    26     .    17
        PITT   24   .    . 13    28    99     .
 [*,*,coils]: FRA DET LAN WIN   STL   FRE   LAF :=
        GARY    .   .   11  .    16     .     8
        CLEV   23   8   10  9    21     .     .
        PITT    .   .    .  .     .    81     . ;
```

Missing parameters like `cost["GARY","FRA","bands"]`, as well as those explicitly
marked "omitted" by use of a dot (like `cost["GARY","FRA","coils"]`), are
given the value 9999. In total, 24 values of 9999 are assigned.

The default feature is especially useful when you want all parameters of an
indexed collection to be assigned the same value. For instance, in Figure 3-2, we apply a
transportation `model` to an assignment problem by setting all supplies and demands to 1.
The `model` declares

```ampl
param supply {ORIG} >= 0;
param demand {DEST} >= 0;
```

but in the `data` we give only a default value:

```ampl
param supply default 1 ;
param demand default 1 ;
```

Since no other values are specified, the default of 1 is automatically assigned to every element
of supply and demand.

As explained in [Chapter 7](../07/07.md), a parameter declaration in the `model` may include a
default expression. This offers an alternative way to specify a single default value:

```ampl
param cost {ORIG,DEST,PROD} >= 0, default 9999;
```

If you just want to avoid storing a lot of 9999's in a `data` file, however, it is better to put
the default phrase in the `data` statement. The default phrase should go in the
`model` when you want the default value to depend in some way on other `data`. For
instance, a different arbitrarily large cost could be given for each product by specifying:

```ampl
param huge_cost {PROD} > 0;
param cost {ORIG, DEST, p in PROD} >= 0, default huge_cost[p];
```

A discussion of default's relation to the = phrase in `param` statements is given in
[Section 7.5](../07/7_5_computed_parameters.ipynb).

Indexed collections of sets
For an indexed collection of sets, separate `data` statements specify the members of
each set in the collection. In the example of [Figure 6-3](../06/6_5_indexed_collections_of_sets.ipynb#fig-6-3), for example, the sets named
AREA are indexed by the set PROD:

```ampl
set PROD;                # products
set AREA {PROD};         # market areas for each product
```

The membership of these sets is given in [Figure 6-4](../06/6_5_indexed_collections_of_sets.ipynb#fig-6-4) by:

```ampl
set PROD := bands coils ;
set AREA[bands] := east north ;
set AREA[coils] := east west export ;
```

Any of the `data` statement formats for a set may be used with indexed collections of sets.
The only difference is that the set name following the keyword set is subscripted.
As for other sets, you may specify one or more members of an indexed collection to
be empty, by giving an empty list of elements. If you want to provide a `data` statement
only for those members of an indexed collection that are not empty, define the empty set
as the default value in the `model`:

```ampl
set AREA {PROD} default {};
```

Otherwise you will be warned about any set whose `data` statement is not provided.

#### Initial values for variables

You may optionally assign initial values to the variables of a `model`, using any of the
options for assigning values to parameters. A variable's name stands for its value, and a
constraint's name stands for the associated dual variable's value. (See Section 12.5 for a
short explanation of dual variables.)
Any `param` `data` statement may specify initial values for variables. The variable or
constraint name is simply used in place of a parameter name, in any of the formats
described by the previous sections of this chapter. To help clarify the intent, the keyword
`var` may be substituted for `param` at the start of a `data` statement. For example, the following
`data` table gives initial values for the variable Trans of [Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a):

```ampl
var Trans:  FRA    DET   LAN   WIN    STL    FRE    LAF :=
      GARY  100    100   800   100    100    500    200
      CLEV  900    100   100   500    500    200    200
      PITT  100    900   100   500    100    900    200 ;
```

As another example, in the `model` of [Figure 1-4](../01/tut_1_4.ipynb#fig-1-4a), a single table can give values for the
parameters rate, profit and market, and initial values for the variables Make:

```ampl
param:   rate  profit market  Make :=
  bands   200    25    6000   3000
  coils   140    30    4000   2500
  plate   160    29    3500   1500 ;
```

All of the previously described features for default values also apply to variables.

Initial values of variables (as well as the values of expressions involving these initial
values) may be viewed before you type `solve`, using the `display`, print or printf
commands described in Sections 12.1 through 12.4. Initial values are also optionally
passed to the solver, as explained in Section 14.1 and A.18.1. After a solution is
returned, the variables no longer have their initial values, but even then you can refer to
the initial values by placing an appropriate suffix after the variable's name, as shown in
Section A.11.

The most common use of initial values is to give a good starting guess to a solver for
nonlinear optimization, which is discussed in Chapter 18.